### This notebook is adapted from [the official Pytorch tutorial on DCGAN](https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html)

In [ ]:
_ROOT_ = '../'

import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.utils as vutils

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from PIL import Image
from IPython.display import HTML

# Set random seed for reproducibility
manualSeed = 999
#manualSeed = random.randint(1, 10000) # use if you want new results
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
torch.use_deterministic_algorithms(True) # Needed for reproducible results

### Choose Your Training Device with `DEVICE_NAME`
CPU only: `'cpu'`, NVIDIA GPU:, `'cuda'`, Apple Silicon: `'mps'`

In [ ]:
DEVICE_NAME = 'mps'
DEVICE = torch.device(DEVICE_NAME)

### Global Varaibles

In [ ]:
# Root directory for dataset
DATA_DIR = _ROOT_ + "./Data/CelebA"

# Batch size during training
BATCH_SIZE = 128

# Size of target images
IMG_SIZE = 32
# Number of channels in the training images. For color images this is 3
IMG_CHANNELS = 3

# dimension of z latent vector (i.e. dimension of generator input)
Z_DIM = 128

# Base number of feature maps in generator
NUM_GFT = 64
# Base number of feature maps in discriminator
NUM_DFT = 64

# Number of training epochs
EPOCHS = 10

# Learning rate for optimizers
LR = 0.0002
# Beta1 hyperparameter for the Adam optimizer
BETA = 0.5

### Dataset and Data Loader

In [ ]:
class CelebA(torch.utils.data.Dataset):
    def __init__(self, img_paths, task):
        self.img_paths = img_paths
        self.imgtrans = self.get_transform(task)
        
    def load_img(self, img_path):
        img = Image.open(img_path, mode="r", formats=None)
        return img
        
    def __len__(self):
        return len(self.img_paths)
        
    def __getitem__(self, index):
        img_path = self.img_paths[index]#
        img = self.load_img(img_path)
        img = self.imgtrans(self.load_img(img_path))
        return img
        
    @staticmethod
    def get_transform(task, image_size=IMG_SIZE):
        if task == 'train':
            transform = transforms.Compose(
                    [
                        transforms.Resize((image_size, image_size)),
                        transforms.ToTensor(),
                        transforms.RandomHorizontalFlip(p=0.5),
                        transforms.Normalize(
                            [0.5 for _ in range(IMG_CHANNELS)],
                            [0.5 for _ in range(IMG_CHANNELS)],
                        ),
                    ]
                )
        else:
            transform = transforms.Compose(
                    [
                        transforms.Resize((image_size, image_size)),
                        transforms.ToTensor(),
                    ]
                )
        return transform

In [ ]:
img_paths = os.listdir(DATA_DIR)
img_paths = [DATA_DIR + '/' + filename for filename in img_paths]

dataset = CelebA(img_paths, task='train')
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Plot some training images
real_batch = next(iter(dataloader))
plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Training Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch.to(DEVICE)[:64], padding=2, normalize=True).cpu(),(1,2,0)))

### Utility Functions

In [ ]:
# custom weights initialization called on the generator and the discriminator
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

### Generator

The generator, $G$, is designed to map the latent space vector
$z$ to data-space. Since our data are images, converting
$z$ to data-space means ultimately creating a RGB image with the
same size as the training images (i.e. 3x32x32). In practice, this is
accomplished through a series of strided two dimensional transpose
convolution layers, each paired with a 2D batch norm layer and a ReLU
activation. The output of the generator is fed through a tanh function
to return it to the input data range of $[-1,1]$. It is worth
noting the existence of the batch norm functions after the
conv-transpose layers, as this is a critical contribution of the DCGAN
paper. These layers help with the flow of gradients during training.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(Z_DIM, NUM_GFT * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NUM_GFT * 8),
            nn.ReLU(True),
            # state size. ``(NUM_GFT*8) x 4 x 4``
            nn.ConvTranspose2d(NUM_GFT * 8, NUM_GFT * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_GFT * 4),
            nn.ReLU(True),
            # state size. ``(NUM_GFT*4) x 8 x 8``
            nn.ConvTranspose2d(NUM_GFT * 4, NUM_GFT * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_GFT * 2),
            nn.ReLU(True),
            # state size. ``(NUM_GFT*2) x 16 x 16``
            nn.ConvTranspose2d(NUM_GFT * 2, NUM_GFT, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_GFT),
            nn.ReLU(True),
            # state size. ``(NUM_GFT) x 32 x 32``
            nn.Conv2d(NUM_GFT, IMG_CHANNELS, 1, 1, 0, bias=False),
            nn.Tanh()
            # state size. ``(IMG_CHANNELS) x 32 x 32``
        )

    def forward(self, z):
        return self.network(z)

### Discriminator

The discriminator, $D$, is a binary classification
network that takes an image as input and outputs a scalar probability
that the input image is real (as opposed to fake). Here, $D$ takes
a 3x32x32 input image, processes it through a series of `Conv2d`,
`BatchNorm2d`, and `LeakyReLU` layers, and outputs the final probability
through a `Sigmoid` activation function. This architecture can be extended
with more layers if necessary for the problem, but there is significance
to the use of the strided convolution, BatchNorm, and LeakyReLUs. The
DCGAN paper mentions it is a good practice to use strided convolution
rather than pooling to downsample because it lets the network learn its
own pooling function. Also batch norm and leaky relu functions promote
healthy gradient flow which is critical for the learning process of both
$G$ and $D$.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            # input is ``(IMG_CHANNELS) x 32 x 32``
            nn.Conv2d(IMG_CHANNELS, NUM_DFT, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(NUM_DFT) x 16 x 16``
            nn.Conv2d(NUM_DFT, NUM_DFT * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_DFT * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(NUM_DFT*2) x 8 x 8``
            nn.Conv2d(NUM_DFT * 2, NUM_DFT * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_DFT * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(NUM_DFT*4) x 4 x 4``
            nn.Conv2d(NUM_DFT * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, imgs):
        # flatten the output to match the requirements of the loss function
        return self.network(imgs).view(-1)

### Testing Models

In [ ]:
test_model = Generator()
z = torch.randn(3, Z_DIM, 1, 1)
fake_imgs = test_model(z)
fake_imgs.shape

In [ ]:
test_model = Discriminator()
imgs = torch.randn(3, 3, 32, 32)
critics = test_model(imgs)
critics.shape

### Loss Functions and Optimizers

With $D$ and $G$ setup, we can specify how they learn
through the loss functions and optimizers. We will use the Binary Cross
Entropy loss
([BCELoss](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html#torch.nn.BCELoss))
function which is defined in PyTorch as:

\begin{align}\ell(x, y) = L = \{l_1,\dots,l_N\}^\top, \quad l_n = - \left[ y_n \cdot \log x_n + (1 - y_n) \cdot \log (1 - x_n) \right], \quad y_n \in \{0, 1\} \end{align}

where $x_n$ is the prediction from the discriminator while $y_n$ is the label for the input sample being real or fake. This loss function is doing the max game in [the original GAN paper by Goodfellow et el.](https://arxiv.org/abs/1406.2661) That is, we are maximizing the expectations below (Equation 1 in the original paper), we want the discriminator to learn to discern real (training data) and fake (generated ones) samples.

\begin{align} \max_{D} \mathbb{E}_{\boldsymbol{x} \sim p_{data}(\boldsymbol{x})}[\log D(\boldsymbol{x})] + \mathbb{E}_{\boldsymbol{z} \sim p_{\boldsymbol{z}}(\boldsymbol{z})} [\log (1 - D(G(\boldsymbol{z})))] \end{align}

The first expectation $\mathbb{E}_{\boldsymbol{x} \sim p_{data}(\boldsymbol{x})}$ encourages the discriminator to classify real samples as 1, while the second expectation $\mathbb{E}_{\boldsymbol{z} \sim p_{\boldsymbol{z}}(\boldsymbol{z})}$ forces the discriminator to classify generated samples as 0.

Next, we define `REAL_LABEL` as 1 and `FAKE_LABEL` as 0. These labels will be used when calculating the losses of $D$ and $G$, and this is also the convention used in the original GAN paper. Finally, we set up two separate optimizers, one for $D$ and one for $G$. As specified in the DCGAN paper, both are Adam optimizers with learning rate 0.0002 and $\beta_1 = 0.5$. 

For keeping track of the generator’s learning progression, we will generate a fixed batch of latent vectors that are drawn from a Gaussian distribution (i.e. `FIXED_NOISE`). In the training loop, we will periodically input this `FIXED_NOISE` into $G$, and over the iterations we will see images form out of the noise.

Note that `FIXED_NOISE` is a 4D tensor instead of 2D. This is because we need two extra dimensions to cater to the transpose convolution, which expetects and produces image-like tensors with shape `(BATCH_SIZE, CHANNEL_NUM, HEIGHT, WIDTH)`.

In [ ]:
# Create the generator
generator = Generator().to(DEVICE)
# Apply the ``weights_init`` function to randomly initialize all weights to ``mean=0``, ``stdev=0.02``.
generator.apply(weights_init)

# Create the Discriminator
discriminator = Discriminator().to(DEVICE)
discriminator.apply(weights_init)

# Initialize the ``BCELoss`` function
criterion = nn.BCELoss()

# Create batch of latent vectors that we will use to visualize
#  the progression of the generator
FIXED_NOISE = torch.randn(64, Z_DIM, 1, 1, device=DEVICE)

# Establish convention for real and fake labels during training
REAL_LABEL = 1.
FAKE_LABEL = 0.

# Setup Adam optimizers for both G and D
optimizerD = optim.Adam(discriminator.parameters(), lr=LR, betas=(BETA, 0.999))
optimizerG = optim.Adam(generator.parameters(), lr=LR, betas=(BETA, 0.999))

### Training

Finally, now that we have all of the parts of the GAN framework defined,
we can train it. Be mindful that training GANs is somewhat of an art
form, as incorrect hyperparameter settings lead to mode collapse with
little explanation of what went wrong. Here, we will closely follow
Algorithm 1 from the original GAN paper, 
while abiding by some of the best
practices shown in [ganhacks](https://github.com/soumith/ganhacks)_.
Namely, we will “construct different mini-batches for real and fake”
images, and also adjust G’s objective function to maximize
$\log(D(G(z)))$. Training is split up into two main parts. Part 1
updates the Discriminator and Part 2 updates the Generator.

**Part 1 - Train the Discriminator**

Recall, the goal of training the discriminator is to maximize the
probability of correctly classifying a given input as real or fake. In
terms of Goodfellow, we wish to “update the discriminator by ascending
its stochastic gradient”. Practically, we want to maximize
$\log(D(x)) + \log(1-D(G(z)))$. Due to the separate mini-batch
suggestion from [ganhacks](https://github.com/soumith/ganhacks)_,
we will calculate this in two steps. First, we
will construct a batch of real samples from the training set, forward
pass through $D$, calculate the loss ($\log(D(x))$), then
calculate the gradients in a backward pass. Secondly, we will construct
a batch of fake samples with the current generator, forward pass this
batch through $D$, calculate the loss ($\log(1-D(G(z)))$),
and *accumulate* the gradients with a backward pass. Now, with the
gradients accumulated from both the all-real and all-fake batches, we
call a step of the Discriminator’s optimizer.

**Part 2 - Train the Generator**

As stated in the original paper, we want to train the Generator by
minimizing $\log(1-D(G(z)))$ in an effort to generate better fakes, namely the min game in GAN, 

\begin{align} \min_{G} \mathbb{E}_{\boldsymbol{z} \sim p_{\boldsymbol{z}}(\boldsymbol{z})} [\log (1 - D(G(\boldsymbol{z})))] \end{align}

However, this was shown by Goodfellow to not provide sufficient
gradients in Section 3 of the original paper: 

> In practice, equation 1 may not provide sufficient gradient for G to learn well. Early in learning, when G is poor, D can reject samples with high confidence because they are clearly different from the training data. In this case, log(1 − D(G(z))) saturates. Rather than training G to minimize log(1 − D(G(z))) we can train G to maximize log D(G(z)).


As a fix, we
instead wish to maximize $\log(D(G(z)))$. In the code we accomplish
this by: classifying the Generator output from Part 1 with the
Discriminator, computing G’s loss *using real labels as GT*, computing
G’s gradients in a backward pass, and finally updating G’s parameters
with an optimizer step. It may seem counter-intuitive to use the real
labels as GT labels for the loss function, but this allows us to use the
$\log(x)$ part of the ``BCELoss`` (rather than the $\log(1-x)$
part) which is exactly what we want.

Finally, we will do some statistic reporting and at the end of each
epoch we will push our fixed_noise batch through the generator to
visually track the progress of G’s training. The training statistics
reported are:

-  **Loss_D** - discriminator loss calculated as the sum of losses for
   the all real and all fake batches ($\log(D(x)) + \log(1 - D(G(z)))$).
-  **Loss_G** - generator loss calculated as $\log(D(G(z)))$
-  **D(x)** - the average output (across the batch) of the discriminator
   for the all real batch. This should start close to 1 then
   theoretically converge to 0.5 when G gets better. Think about why
   this is. (check Section 4.1 in the original paper for answer)
-  **D(G(z))** - average discriminator outputs for the all fake batch.
   The first number is before D is updated and the second number is
   after D is updated. These numbers should start near 0 and converge to
   0.5 as G gets better. Think about why this is. (check Section 4.1 in the original paper for answer)

**Note:** This step might take a while, depending on how many epochs you
run and if you removed some data from the dataset.

In [ ]:
# Training Loop

# Lists to keep track of progress
img_list = []
G_losses = []
D_losses = []
iters = 0

print("Starting Training Loop...")
# For each epoch
for epoch in range(EPOCHS):
    # For each batch in the dataloader
    for i, data in enumerate(dataloader):
        
        ############################
        # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
        ###########################
        ## Train with all-real batch
        discriminator.zero_grad()
        # Format batch
        real_samples = data.to(DEVICE) 
        b_size = real_samples.shape[0]
        label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=DEVICE)
        # Forward pass real batch through D
        output = discriminator(real_samples).view(-1)
        # Calculate loss on all-real batch
        errD_real = criterion(output, label)
        # Calculate gradients for D in backward pass
        errD_real.backward()
        D_x = output.mean().item()

        ## Train with all-fake batch
        # Generate batch of latent vectors
        noise = torch.randn(b_size, Z_DIM, 1, 1, device=DEVICE)
        # Generate fake image batch with G
        fake = generator(noise)
        label.fill_(FAKE_LABEL)
        # Classify all fake batch with D
        output = discriminator(fake.detach()).view(-1)
        # Calculate D's loss on the all-fake batch
        errD_fake = criterion(output, label)
        # Calculate the gradients for this batch, accumulated (summed) with previous gradients
        errD_fake.backward()
        D_G_z1 = output.mean().item()
        # Compute error of D as sum over the fake and the real batches
        errD = errD_real + errD_fake
        # Update D
        optimizerD.step()

        ############################
        # (2) Update G network: maximize log(D(G(z)))
        ###########################
        generator.zero_grad()
        label.fill_(REAL_LABEL)  # fake labels are real for generator cost
        # Since we just updated D, perform another forward pass of all-fake batch through D
        output = discriminator(fake).view(-1)
        # Calculate G's loss based on this output
        errG = criterion(output, label)
        # Calculate gradients for G
        errG.backward()
        D_G_z2 = output.mean().item()
        # Update G
        optimizerG.step()
        
        # Output training stats
        if i % 500 == 0:
            print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
                  % (epoch, EPOCHS, i, len(dataloader),
                     errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))
        
        # Save Losses for plotting later
        G_losses.append(errG.item())
        D_losses.append(errD.item())
        
        # Check how the generator is doing by saving G's output on fixed_noise
        if (iters % 500 == 0) or ((epoch == EPOCHS-1) and (i == len(dataloader)-1)):
            with torch.no_grad():
                fake = generator(FIXED_NOISE).detach().cpu()
            img_list.append(vutils.make_grid(fake, padding=2, normalize=True))
            
        iters += 1

In [ ]:
torch.save(generator.state_dict(), "./generator.pth")
print("Saved Generator to generator.pth")

torch.save(discriminator.state_dict(), "./discriminator.pth")
print("Saved Discriminator to discriminator.pth")

### Visualization of G’s progression

In [ ]:
fig = plt.figure(figsize=(8,8))
plt.axis("off")
ims = [[plt.imshow(np.transpose(i,(1,2,0)), animated=True)] for i in img_list]
ani = animation.ArtistAnimation(fig, ims, interval=1000, repeat_delay=1000, blit=True)

HTML(ani.to_jshtml())

### Real Images vs. Fake Images

In [ ]:
# Grab a batch of real images from the dataloader
real_batch = next(iter(dataloader))

# Plot the real images
plt.figure(figsize=(15,15))
plt.subplot(1,2,1)
plt.axis("off")
plt.title("Real Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch.to(DEVICE)[:64], padding=5, normalize=True).cpu(),(1,2,0)))

# Plot the fake images from the last epoch
plt.subplot(1,2,2)
plt.axis("off")
plt.title("Fake Images")
plt.imshow(np.transpose(img_list[-1],(1,2,0)))
plt.show()

### Load a Pretrained Generator for Image Synthesis

In [ ]:
model = Generator().to(DEVICE)
model.load_state_dict(torch.load(_ROOT_ + "./Model/DCGAN.pth", map_location=DEVICE))

# Generate batch of latent vectors
latent = torch.randn(b_size, Z_DIM, 1, 1, device=DEVICE)
# Generate fake image batch with G
image = model(latent)[:64].detach().cpu()
imagegrid = vutils.make_grid(image, padding=2, normalize=True)

plt.axis("off")
plt.title("Synthetic Images")
plt.imshow(np.transpose(imagegrid, (1,2,0)))
plt.show()